# A positive exit surplus can still hide a cash shortfall

**DevExit cash-buffer notebook | Matt Lenzie | 23 November 2026 | DX-14**

This fictional example ends with more cash than it started with, yet needs an extra **£12,750 before the first sale receipt**. Looking only at month-end balances misses that gap. Delaying every sale by one month increases the gap to **£28,000**.

The [development exit finance costs guide at DevExit.co.uk](https://devexit.co.uk/guides/development-exit-finance-costs/) explains the separate cost components and the distinction between gross and net finance. This notebook adds a different check: when cash leaves the business compared with when the business can use retained sale proceeds.

Read the saved results without running code. To change assumptions, make your own copy, edit the inputs and run all cells from the top. Colab may require a Google sign-in to run or save a copy. The calculation uses Python's standard library, does not fetch data, and does not request credentials or connect to Drive.

All amounts and timing assumptions are invented. Prepared with AI assistance; calculations tested locally. This is an illustrative planning tool, not an assessment of an actual scheme.


## 1. The cash that is available matters

The model begins **after the exit facility has completed**. Opening unrestricted cash is £45,000 and opening principal is £1.5 million. These are separate balances: do not enter the gross facility as available cash. Use the earlier [net-advance calculator](https://mattylll.github.io/devexit-2026-q3-08-development-exit-finance-rates/net-advance/) for the completion-day funding bridge.

Three £600,000 sales complete at the end of months 3, 5 and 7. Each has £12,000 selling costs and an assumed £500,000 principal release, leaving £88,000 for the cash account. The model assumes that retained amount is immediately available without restriction. These release amounts are arithmetic assumptions, not evidence of lender consent. The [final-unit release calculator](https://devexit-final-unit-releases.streamlit.app/) explores the separate relationship between releases, retained cash and remaining debt.

For this example only, interest is **serviced monthly at an assumed 0.75% of opening monthly principal**, with £4,000 of holding costs each month. Another £12,000 of cash-paid costs falls in month 1. Costs are paid **before** that month's sale receipt; releases reduce the principal charged in the following month. There is no retained or rolled interest, compounding, minimum-interest charge, tax or exit fee in this example. Add relevant cash items once, at their due dates. Do not count a fee twice if it has already reduced opening cash.

The horizon runs through the last sale. Costs continue through that month. A sale and release do not automatically end every real obligation. Do not interpret the closing cash as development profit: original land, build and equity cash flows are outside this model.


## 2. What the ordering changes

| Fictional scenario | Minimum opening cash needed | Extra above £45,000 available | Lowest cash before a sale receipt | Cash at end of horizon |
| :--- | ---: | ---: | ---: | ---: |
| Sales in months 3, 5 and 7 | £57,750 | £12,750 | -£12,750 | £212,750 |
| All sales one month later | £73,000 | £28,000 | -£28,000 | £197,500 |

In the base case, month 2 closes with £2,500. Month 3 outgoings of £15,250 take cash to **-£12,750 before** its £88,000 retained sale receipt arrives. Month 3 then closes at £75,250. Every month-end balance is positive, but those snapshots have hidden the earlier shortfall.

Negative balances represent an **unfunded gap**, not an authorised overdraft. The later receipt does not fund a payment that was already due. Minimum opening cash is the largest cumulative deficit across the ordered events, starting from zero. The additional requirement is the excess of that minimum over cash actually available. It contains no contingency margin.

Moving all receipts one month later also keeps the full £1.5 million principal outstanding for an extra month. In this model the additional £11,250 interest plus £4,000 holding cost raises the minimum cash requirement by £15,250. Both scenarios repay principal in full; the cash result alone does not test value cover or other lending conditions.


In [1]:
#@title Calculation: expand to inspect the event ordering
from decimal import Decimal, ROUND_HALF_UP
from math import isfinite
from copy import deepcopy
import csv
import io

def money(value):
    amount = Decimal(str(value))
    if not amount.is_finite() or amount < 0:
        raise ValueError("Amounts must be finite and non-negative")
    return amount.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)

def cash_ledger(opening_cash, principal, monthly_rate, holding_cost,
                months, sales, extra_costs=None):
    """Pay monthly costs first, then retain net sale cash and reduce debt."""
    if type(months) is not int or months < 1:
        raise ValueError("months must be a positive integer")
    rate = Decimal(str(monthly_rate))
    if not rate.is_finite() or not 0 <= rate <= 1:
        raise ValueError("monthly_rate must be a fraction from 0 to 1")
    cash, debt, holding = map(money, (opening_cash, principal, holding_cost))
    initial = cash
    extra_costs = extra_costs or {}
    for month, amount in extra_costs.items():
        if type(month) is not int or not 1 <= month <= months:
            raise ValueError("Extra-cost month outside horizon")
        money(amount)
    grouped = {}
    for sale in sales:
        month = sale['month']
        if type(month) is not int or not 1 <= month <= months:
            raise ValueError("Sale month outside horizon")
        values = tuple(money(sale[k]) for k in ('gross', 'selling_cost', 'release'))
        grouped.setdefault(month, []).append(values)
    low_cash = cash
    low_month_end = cash
    rows = []
    for month in range(1, months + 1):
        opening_debt = debt
        interest = (debt * rate).quantize(Decimal('0.01'), rounding=ROUND_HALF_UP)
        outgoing = interest + holding + money(extra_costs.get(month, 0))
        cash -= outgoing
        before_sales = cash
        low_cash = min(low_cash, cash)
        retained = Decimal('0.00')
        for gross, selling_cost, release in grouped.get(month, []):
            if release > debt:
                raise ValueError("Principal release exceeds outstanding debt")
            receipt = gross - selling_cost - release
            debt -= release
            retained += receipt
            cash += receipt
            low_cash = min(low_cash, cash)
        low_month_end = min(low_month_end, cash)
        rows.append(dict(month=month, opening_debt=opening_debt,
                         interest=interest, cash_outgoing=outgoing,
                         before_sales=before_sales, retained_receipt=retained,
                         closing_cash=cash, closing_debt=debt))
    minimum_opening = max(Decimal('0.00'), initial - low_cash)
    return dict(rows=rows, minimum_opening=minimum_opening,
                additional_cash=max(Decimal('0.00'), -low_cash),
                lowest_cash=low_cash, lowest_month_end=low_month_end,
                closing_cash=cash, closing_debt=debt)


In [2]:
#@title Editable assumptions: base and one-month delay
# Edit this cell in your own copy. Rates are fractions: 0.0075 = 0.75%.
inputs = dict(
    opening_cash=45000,
    principal=1500000,
    monthly_rate=0.0075,
    holding_cost=4000,
    months=7,
    sales=[dict(month=m, gross=600000, selling_cost=12000, release=500000)
           for m in (3, 5, 7)],
    extra_costs={1: 12000},
)
base = cash_ledger(**inputs)
delayed_inputs = deepcopy(inputs)
delayed_inputs['months'] += 1
for sale in delayed_inputs['sales']:
    sale['month'] += 1
delayed = cash_ledger(**delayed_inputs)
print('Base and one-month-delay scenarios calculated.')


Base and one-month-delay scenarios calculated.


In [3]:
#@title Calculated results and monthly cash ledger
def gbp(value):
    return f"GBP {value:,.2f}"

def report(label, result):
    print(label)
    print('Minimum opening cash:', gbp(result['minimum_opening']))
    print('Extra cash required: ', gbp(result['additional_cash']))
    print('Lowest event cash:   ', gbp(result['lowest_cash']))
    print('Lowest month-end:    ', gbp(result['lowest_month_end']))
    print('Closing cash:        ', gbp(result['closing_cash']))
    print('Closing debt:        ', gbp(result['closing_debt']))
    print('Month | Outgoings | Before sales | Retained | Closing cash')
    for row in result['rows']:
        print(f"{row['month']:>5} | {row['cash_outgoing']:>9,.0f} | "
              f"{row['before_sales']:>12,.0f} | {row['retained_receipt']:>8,.0f} | "
              f"{row['closing_cash']:>12,.0f}")
    print()

report('BASE CASE: amounts in GBP', base)
report('ALL SALES ONE MONTH LATER: amounts in GBP', delayed)


BASE CASE: amounts in GBP
Minimum opening cash: GBP 57,750.00
Extra cash required:  GBP 12,750.00
Lowest event cash:    GBP -12,750.00
Lowest month-end:     GBP 2,500.00
Closing cash:         GBP 212,750.00
Closing debt:         GBP 0.00
Month | Outgoings | Before sales | Retained | Closing cash
    1 |    27,250 |       17,750 |        0 |       17,750
    2 |    15,250 |        2,500 |        0 |        2,500
    3 |    15,250 |      -12,750 |   88,000 |       75,250
    4 |    11,500 |       63,750 |        0 |       63,750
    5 |    11,500 |       52,250 |   88,000 |      140,250
    6 |     7,750 |      132,500 |        0 |      132,500
    7 |     7,750 |      124,750 |   88,000 |      212,750

ALL SALES ONE MONTH LATER: amounts in GBP
Minimum opening cash: GBP 73,000.00
Extra cash required:  GBP 28,000.00
Lowest event cash:    GBP -28,000.00
Lowest month-end:     GBP -12,750.00
Closing cash:         GBP 197,500.00
Closing debt:         GBP 0.00
Month | Outgoings | Before sales 

## 3. Use the result as a question for the plan

The extra £12,750 in the base case is not a suggestion to borrow that exact amount. It is the uncovered cash requirement under this sequence. Establish which unrestricted funds can meet it, when they are available, and what buffer is needed if assumptions change. A real plan needs dated payments and receipts, not just month numbers. Contractual restrictions, taxes, delays in releasing proceeds and costs outside the horizon can increase the requirement.

This is a simple serviced-interest case. A facility with retained or rolled interest needs a different cash and debt schedule. The [minimum-interest comparison](https://medium.com/@matthewalenzie/a-lower-exit-finance-rate-can-cost-more-on-an-early-sale-cdbfc0679557) shows why an actual repayment date can change cost beyond a simple monthly calculation.

If refinance is the fallback, the cash ledger must run until replacement funds are actually available. The [refinance-start timetable](https://telegra.ph/The-refinance-start-date-comes-before-loan-maturity-11-23) works backwards through preparation, valuation, legal work and completion. An expected funding date is not money in the account.

Use the [DevExit research and tools guide](https://sites.google.com/view/devexit-north-west-exit-routes/home/research-and-tools) to keep the separate evidence, available-advance, principal-release and timing checks together. These linked publications share authorship and commercial affiliation; they are not independent corroborating studies.


In [4]:
#@title Reproducibility checks: fixed fixtures, independent of edited assumptions
# These fixed fixtures validate the model independently of your edited inputs.
fixture = dict(opening_cash=45000, principal=1500000, monthly_rate=0.0075,
               holding_cost=4000, months=7,
               sales=[dict(month=m, gross=600000, selling_cost=12000, release=500000)
                      for m in (3, 5, 7)], extra_costs={1: 12000})
check = cash_ledger(**fixture)
assert check['minimum_opening'] == Decimal('57750.00')
assert check['additional_cash'] == Decimal('12750.00')
assert check['lowest_month_end'] == Decimal('2500.00')
assert check['closing_cash'] == Decimal('212750.00')
assert check['closing_debt'] == 0
later = deepcopy(fixture)
later['months'] = 8
for sale in later['sales']:
    sale['month'] += 1
later_check = cash_ledger(**later)
assert later_check['minimum_opening'] == Decimal('73000.00')
assert later_check['additional_cash'] == Decimal('28000.00')
assert later_check['closing_cash'] == Decimal('197500.00')
funded = cash_ledger(**dict(fixture, opening_cash=57750))
assert funded['lowest_cash'] == 0
assert funded['additional_cash'] == 0
zero = cash_ledger(0, 0, 0, 0, 1, [])
assert zero['minimum_opening'] == 0
no_sales = cash_ledger(0, 1000, 0.01, 0, 2, [])
assert no_sales['minimum_opening'] == 20
assert no_sales['closing_debt'] == 1000
top_up = cash_ledger(0, 100, 0, 0, 1,
                    [dict(month=1, gross=80, selling_cost=0, release=100)])
assert top_up['additional_cash'] == 20
for invalid in (dict(fixture, monthly_rate=-0.1),
                dict(fixture, sales=[dict(month=8, gross=1, selling_cost=0, release=0)]),
                dict(fixture, sales=[dict(month=1, gross=2000000, selling_cost=0, release=1600000)])):
    try:
        cash_ledger(**invalid)
        raise AssertionError('Invalid input was not rejected')
    except ValueError:
        pass
print('Checks passed: base, delayed, exact funding, zero-cost, no-sales, sale top-up and invalid inputs.')


Checks passed: base, delayed, exact funding, zero-cost, no-sales, sale top-up and invalid inputs.


In [5]:
#@title CSV ledger: copy the output to save your scenario
# Produces CSV text for the edited base scenario. No file upload or network call.
csv_buffer = io.StringIO()
writer = csv.DictWriter(csv_buffer, fieldnames=list(base['rows'][0]))
writer.writeheader()
writer.writerows(base['rows'])
print(csv_buffer.getvalue())


month,opening_debt,interest,cash_outgoing,before_sales,retained_receipt,closing_cash,closing_debt
1,1500000.00,11250.00,27250.00,17750.00,0.00,17750.00,1500000.00
2,1500000.00,11250.00,15250.00,2500.00,0.00,2500.00,1500000.00
3,1500000.00,11250.00,15250.00,-12750.00,88000.00,75250.00,1000000.00
4,1000000.00,7500.00,11500.00,63750.00,0.00,63750.00,1000000.00
5,1000000.00,7500.00,11500.00,52250.00,88000.00,140250.00,500000.00
6,500000.00,3750.00,7750.00,132500.00,0.00,132500.00,500000.00
7,500000.00,3750.00,7750.00,124750.00,88000.00,212750.00,0.00



## Method and rights

Version 1.0, 23 November 2026. Original hypothetical arithmetic, not observed lending data. Every cash charge is rounded to the nearest penny using decimal half-up rounding. The model checks the cash balance after monthly outgoings and after each listed sale. For multiple sales in one month, their listed order controls the cash sequence. Results are conditional on that order; no daily-interest calculation or uncertain-date simulation is implied.

Original commentary and code © 2026 DevExit. All rights reserved. You may run this notebook and make a personal working copy to explore the illustrated calculation; no blanket open redistribution licence is granted. No Rightmove series or third-party dataset is reproduced. Linked source material retains its own rights.

DevExit is a trading name of Lenzie Consulting Ltd. We are a finance arranger and introducer, not a lender. Lenzie Consulting Ltd is not authorised or regulated by the Financial Conduct Authority. We do not provide financial, legal, tax or investment advice. This is general illustrative material, not a lender quote, finance offer, release approval or recommendation. Any actual facility is subject to its agreed terms, status, valuation and lender criteria.
